<div style="background-color:#000047; padding: 30px; border-radius: 10px; color: white; text-align: center;">
    <img src='Figures/alinco_white_text.png' style="height: 100px; margin-bottom: 10px;"/>
    <h1>Módulo 1: Preprocesamiento y Representación de Texto</h1>
    <h2>Word Embeddings</h2>
</div>


## 1. ¿Por qué Embeddings? 

Los embeddings resuelven las principales limitaciones de BoW/TF-IDF.

#### El problema de BoW y TF-IDF

```
"rey"   → [1, 0, 0, 0, 0, ...]  One-Hot: no hay similitud entre palabras
"reina" → [0, 1, 0, 0, 0, ...]
"perro" → [0, 0, 1, 0, 0, ...]

similitud("rey", "reina") = 0   ❌  (debería ser alto)
similitud("rey", "perro") = 0   ❌  (debería ser bajo)
```

#### La solución: Embeddings densos

```
"rey"   → [0.91, 0.32, -0.45, ..., 0.12]  (300 dimensiones, denso)
"reina" → [0.88, 0.29, -0.41, ..., 0.09]
"perro" → [0.05, -0.73, 0.88, ..., -0.42]

similitud("rey", "reina") = 0.89  ✅
similitud("rey", "perro") = 0.12  ✅
```

#### La Hipótesis Distribucional

> *"You shall know a word by the company it keeps"* — J.R. Firth (1957)

Palabras que aparecen en contextos similares tienen significados similares.

```
"El gato maulla"    -> gato aparece cerca de: maulla, ronronea, duerme
"El perro ladra"    -> perro aparece cerca de: ladra, corre, juega
"El felino maulla"  -> felino aparece cerca de: maulla, ronronea

=> gato y felino tienen contextos similares => vectores similares!
```
#### Propiedades de los embeddings:
```
v(rey) - v(hombre) + v(mujer) ≈ v(reina)  [analogia de genero]
v(Paris) - v(Francia) + v(Espana) ≈ v(Madrid)  [analogia de capitales]
v(mayor) - v(grande) + v(pequeno) ≈ v(menor)  [analogia de tamano]
```
### Evolución de los Embeddings

| Modelo | Año | Tipo | Ventaja principal |
|---|---|---|---|
| **LSA** | 1990 | Matricial | Primer embedding semántico |
| **Word2Vec** | 2013 | Red neuronal superficial | Rápido, analogías |
| **GloVe** | 2014 | Co-ocurrencias globales | Combina local y global |
| **FastText** | 2016 | Subword | Maneja OOV, morfología |
| **ELMo** | 2018 | LSTM bidireccional | Contextual |
| **BERT** | 2018 | Transformer | Bidireccional, estado del arte |
| **Sentence-BERT** | 2019 | BERT + siamese | Oraciones eficientes |

## 2. Word2Vec: CBOW y Skip-gram 

### Arquitecturas de Word2Vec

**CBOW (Continuous Bag of Words):** predice la palabra central a partir de su contexto.

```
Contexto: ["el", "come", "fresco"]
Predice:  "gato"
```

- Predice la palabra central dado su contexto (palabras vecinas)
- Mas rapido de entrenar
- Mejor para palabras frecuentes
```
Contexto:      [El, __, en, el, parque]
                       |
                       v  PREDICE
Palabra central:    perro
```
[Efficient Estimation of Word Representations in
Vector Space](https://arxiv.org/pdf/1301.3781)



**Skip-gram:** predice el contexto a partir de la palabra central.

```
Palabra:  "gato"
Predice:  ["el", "come", "fresco"]
```
- Predice el contexto dado la palabra central
- Mas lento pero mejor para palabras raras
- Mejor para corpus pequenos
```
Palabra central:    perro
                       |
                       v  PREDICE
Contexto:      [El, corre, en, el, parque]
```


| | CBOW | Skip-gram |
|---|---|---|
| Velocidad | Más rápido | Más lento |
| Palabras frecuentes | Mejor | Similar |
| Palabras raras | Regular | **Mejor** |
| Corpus pequeño | Mejor | Similar |

[TensorFlow Word2Vec](https://projector.tensorflow.org/)

[word2vec](https://ameersaleem.substack.com/p/explaining-word-embeddings-with-word2vec)

[CBOW & Skip-Gram](https://towardsdatascience.com/word2vec-explained-49c52b4ccb71/)

In [13]:
import numpy as np
import re
from collections import defaultdict
from NLP_TextCleaner import TextCleaningPipeline

#  WORD2VEC SIMPLIFICADO 

# Mini corpus
corpus_mini = [
    "el gato come pescado",
    "el perro come carne",
    "el gato y el perro son animales",
    "el pescado y la carne son alimentos",
    "los animales necesitan alimentos para vivir",
]

# 1.- Preparar tokens y vocabulario
pipeline_es = TextCleaningPipeline(idioma='', remover_stops = True, metodo='lema')
tokens = [pipeline_es.procesar(doc)['tokens'] for doc in corpus_mini]

vocab = sorted(set(t for doc in tokens for t in doc))
w2i = {w:i for i, w in enumerate(vocab)}
i2w = {i:w for w, i in w2i.items()}
V = len(vocab)

# 2.- Generar pares (centro, contexto) con ventana=2
def generar_pares(tokens_corpus, ventana = 2):
    pares = []
    for doc in tokens_corpus:
        for i, centro in enumerate(doc):
            for j in range(max(0, i-ventana), min(len(doc), i+ventana+1)):
                if j!=i:
                    pares.append((centro, doc[j]))
    return pares

pares = generar_pares(tokens)
pares                
            

[('el', 'gato'),
 ('el', 'come'),
 ('gato', 'el'),
 ('gato', 'come'),
 ('gato', 'pescado'),
 ('come', 'el'),
 ('come', 'gato'),
 ('come', 'pescado'),
 ('pescado', 'gato'),
 ('pescado', 'come'),
 ('el', 'perro'),
 ('el', 'come'),
 ('perro', 'el'),
 ('perro', 'come'),
 ('perro', 'carne'),
 ('come', 'el'),
 ('come', 'perro'),
 ('come', 'carne'),
 ('carne', 'perro'),
 ('carne', 'come'),
 ('el', 'gato'),
 ('el', 'el'),
 ('gato', 'el'),
 ('gato', 'el'),
 ('gato', 'perro'),
 ('el', 'el'),
 ('el', 'gato'),
 ('el', 'perro'),
 ('el', 'son'),
 ('perro', 'gato'),
 ('perro', 'el'),
 ('perro', 'son'),
 ('perro', 'animales'),
 ('son', 'el'),
 ('son', 'perro'),
 ('son', 'animales'),
 ('animales', 'perro'),
 ('animales', 'son'),
 ('el', 'pescado'),
 ('el', 'la'),
 ('pescado', 'el'),
 ('pescado', 'la'),
 ('pescado', 'carne'),
 ('la', 'el'),
 ('la', 'pescado'),
 ('la', 'carne'),
 ('la', 'son'),
 ('carne', 'pescado'),
 ('carne', 'la'),
 ('carne', 'son'),
 ('carne', 'alimentos'),
 ('son', 'la'),
 ('son', '

In [4]:
tokens

[['el', 'gato', 'come', 'pescado'],
 ['el', 'perro', 'come', 'carne'],
 ['el', 'gato', 'el', 'perro', 'son', 'animales'],
 ['el', 'pescado', 'la', 'carne', 'son', 'alimentos'],
 ['los', 'animales', 'necesitan', 'alimentos', 'para', 'vivir']]

In [8]:
vocab

['alimentos',
 'animales',
 'carne',
 'come',
 'el',
 'gato',
 'la',
 'los',
 'necesitan',
 'para',
 'perro',
 'pescado',
 'son',
 'vivir']

In [10]:
w2i

{'alimentos': 0,
 'animales': 1,
 'carne': 2,
 'come': 3,
 'el': 4,
 'gato': 5,
 'la': 6,
 'los': 7,
 'necesitan': 8,
 'para': 9,
 'perro': 10,
 'pescado': 11,
 'son': 12,
 'vivir': 13}

In [11]:
i2w

{0: 'alimentos',
 1: 'animales',
 2: 'carne',
 3: 'come',
 4: 'el',
 5: 'gato',
 6: 'la',
 7: 'los',
 8: 'necesitan',
 9: 'para',
 10: 'perro',
 11: 'pescado',
 12: 'son',
 13: 'vivir'}

In [15]:
# Embeddings por co-ocurrencias (versión simplificada GloVe)
# Construimos una matriz de co-ocurrencias y luego aplicamos SVD

# Matriz de co-ocurrencias
cooc = np.zeros((V,V))
for centro, contexto in pares:
    cooc[w2i[centro], w2i[contexto]] +=1
cooc

array([[0., 1., 1., 0., 0., 0., 0., 0., 1., 1., 0., 0., 1., 1.],
       [1., 0., 0., 0., 0., 0., 0., 1., 1., 0., 1., 0., 1., 0.],
       [1., 0., 0., 1., 0., 0., 1., 0., 0., 0., 1., 1., 1., 0.],
       [0., 0., 1., 0., 2., 1., 0., 0., 0., 0., 1., 1., 0., 0.],
       [0., 0., 0., 2., 2., 3., 1., 0., 0., 0., 2., 1., 1., 0.],
       [0., 0., 0., 1., 3., 0., 0., 0., 0., 0., 1., 1., 0., 0.],
       [0., 0., 1., 0., 1., 0., 0., 0., 0., 0., 0., 1., 1., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.],
       [1., 1., 0., 0., 0., 0., 0., 1., 0., 1., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 1.],
       [0., 1., 1., 1., 2., 1., 0., 0., 0., 0., 0., 0., 1., 0.],
       [0., 0., 1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0.],
       [1., 1., 1., 0., 1., 0., 1., 0., 0., 0., 1., 0., 0., 0.],
       [1., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0.]])

In [16]:
vocab

['alimentos',
 'animales',
 'carne',
 'come',
 'el',
 'gato',
 'la',
 'los',
 'necesitan',
 'para',
 'perro',
 'pescado',
 'son',
 'vivir']

In [17]:
# SVD para obtener embeddings de baja dimensión
from numpy.linalg import svd

 **SVD**: La Descomposición en Valores Singulares (por sus siglas en inglés, Singular Value Decomposition) es una técnica fundamental para factorizar matrices, utilizada en reducción de dimensionalidad, compresión de imágenes y sistemas de recomendación.

In [19]:
help(svd)

Help on _ArrayFunctionDispatcher in module numpy.linalg:

svd(a, full_matrices=True, compute_uv=True, hermitian=False)
    Singular Value Decomposition.
    
    When `a` is a 2D array, and ``full_matrices=False``, then it is
    factorized as ``u @ np.diag(s) @ vh = (u * s) @ vh``, where
    `u` and the Hermitian transpose of `vh` are 2D arrays with
    orthonormal columns and `s` is a 1D array of `a`'s singular
    values. When `a` is higher-dimensional, SVD is applied in
    stacked mode as explained below.
    
    Parameters
    ----------
    a : (..., M, N) array_like
        A real or complex array with ``a.ndim >= 2``.
    full_matrices : bool, optional
        If True (default), `u` and `vh` have the shapes ``(..., M, M)`` and
        ``(..., N, N)``, respectively.  Otherwise, the shapes are
        ``(..., M, K)`` and ``(..., K, N)``, respectively, where
        ``K = min(M, N)``.
    compute_uv : bool, optional
        Whether or not to compute `u` and `vh` in addition to `

In [18]:
U, S, Vt = svd(cooc, full_matrices = False)


In [21]:
len(U), len(S), len(Vt)

(14, 14, 14)

In [25]:
U.shape, S.shape, Vt.shape

((14, 14), (14,), (14, 14))

In [22]:
# dimensión del embedding
DIM = 3
# primeras DIM componentes
embeddings = U[:,:DIM]*S[:DIM]
embeddings

array([[-0.57611403, -1.84261952,  0.71053874],
       [-0.69934607, -1.4791639 , -0.82117827],
       [-1.42449299, -0.77883308, -1.29966538],
       [-2.46087246,  0.35845327,  0.77694226],
       [-4.50018431,  0.71789386, -1.55273178],
       [-2.83927363,  0.68104537,  0.9058882 ],
       [-1.29435705, -0.25413484,  0.79520077],
       [-0.12819036, -0.80281181,  0.27526323],
       [-0.21441063, -1.44282121,  0.00568965],
       [-0.12469491, -1.12682491, -0.18147973],
       [-2.52095879, -0.23237643,  1.07112421],
       [-1.75630809,  0.19903476,  0.12636494],
       [-1.54535113, -1.06306725,  0.37018838],
       [-0.09831605, -0.8158512 , -0.17858065]])

In [27]:
# Normalizar
norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
embeddings_norm = embeddings/(norms + 1e-8)
embeddings_norm

array([[-0.28004939, -0.89569849,  0.34539332],
       [-0.38201692, -0.80799144, -0.44856761],
       [-0.68497183, -0.37450428, -0.6249481 ],
       [-0.94453381,  0.13758179,  0.29820653],
       [-0.93474309,  0.1491153 , -0.3225213 ],
       [-0.92874388,  0.22277413,  0.29632161],
       [-0.84036953, -0.16499866,  0.51628915],
       [-0.1493507 , -0.93533166,  0.32070083],
       [-0.14698983, -0.98913031,  0.00390056],
       [-0.10860632, -0.98143785, -0.15806455],
       [-0.91707378, -0.08453384,  0.38965331],
       [-0.99111026,  0.11231822,  0.07130958],
       [-0.80829137, -0.55603421,  0.19362594],
       [-0.1169129 , -0.97017248, -0.21235985]])

In [ ]:
#  Similitud de coseno con embeddings propios 
def similitud_coseno(a, b):
    return 

def palabras_mas_similares(palabra, embeddings, w2i, i2w, top_n=4):
    
    return 

print("SIMILITUD SEMÁNTICA :")


## 3. Word2Vec con Gensim

📚 Documentación: https://radimrehurek.com/gensim/models/word2vec.html

In [ ]:
from gensim.models import Word2Vec
import re

# Corpus de entrenamiento 
textos_nlp = [
    "el procesamiento de lenguaje natural usa modelos de machine learning",
    "los modelos de deep learning aprenden representaciones del lenguaje",
    "bert y gpt son modelos de transformers para el lenguaje natural",
    "word2vec aprende embeddings de palabras a partir de grandes corpus",
    "la tokenización divide el texto en palabras y subpalabras",
    "spacy y nltk son las librerías más usadas para el procesamiento de texto",
    "los embeddings capturan el significado semántico de las palabras",
    "el análisis de sentimientos clasifica textos en positivos y negativos",
    "la similitud de coseno mide la cercanía entre vectores de palabras",
    "los modelos de lenguaje generan texto coherente y fluido",
    "bert usa atención bidireccional para entender el contexto completo",
    "gpt genera texto de izquierda a derecha con atención causal",
    "fasttext maneja palabras fuera de vocabulario usando subpalabras",
    "glove aprende embeddings globales usando matrices de co-ocurrencias",
    "los sentence transformers generan embeddings para oraciones completas",
    "la extracción de entidades nombradas identifica personas lugares y organizaciones",
    "el etiquetado de partes del discurso asigna categorías gramaticales",
    "los modelos pre-entrenados se adaptan con fine-tuning a tareas específicas",
    "scikit-learn implementa algoritmos de clasificación y regresión",
    "pandas facilita la manipulación y análisis de datos tabulares",
]

# Tokenizar
sentences = [[re.sub(r'[^a-záéíóúüñ]', '', t) for t in doc.split()] for doc in textos_nlp]
sentences = [[t for t in doc if t] for doc in sentences]

# Entrenar Word2Vec 



In [ ]:
# Palabras más similares 


In [ ]:
# Similitud entre pares de palabras
pares_sim = [
    ('bert', 'gpt'),
    ('bert', 'tokenización'),
    ('embeddings', 'vectores'),
    ('modelos', 'algoritmos'),
    ('lenguaje', 'texto'),
]

print("SIMILITUD ENTRE PARES DE PALABRAS:")


In [ ]:
# Visualización t-SNE de embeddings 
# El t-SNE (Incrustación Estocástica de Vecinos Distribuidos en t) es un algoritmo avanzado de aprendizaje no supervisado 
# diseñado para reducir la dimensionalidad de los datos y permitir su visualización

import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import numpy as np

words_to_plot = [
    'bert', 'gpt', 'transformers', 'modelos', 'deep',
    'embeddings', 'vectores', 'lenguaje', 'texto', 'palabras',
    'tokenización', 'procesamiento', 'clasificación', 'sentimientos',
    'spacy', 'nltk', 'fasttext', 'glove',
]

# t-SNE para reducir a 2D

# Plotear palabras reducidas
plt.figure(figsize=(10, 7))
plt.scatter(coords[:, 0], coords[:, 1], c='steelblue', s=100, alpha=0.7)

for i, word in enumerate(words_to_plot):
    plt.annotate(word, coords[i],
                 textcoords='offset points', xytext=(5, 5), fontsize=9)

plt.title('Visualización t-SNE de Word Embeddings (Word2Vec)', fontsize=12)
plt.xlabel('Dimensión 1 (t-SNE)')
plt.ylabel('Dimensión 2 (t-SNE)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. GloVe y FastText 

### GloVe (Global Vectors for Word Representation)

Desarrollado por Pennington et al. (Stanford, 2014). Combina:
- **Conteo global** de co-ocurrencias (matriz de co-ocurrencia)
- **Optimizacion local** similar a Word2Vec

**GloVe** combina la información de co-ocurrencias **globales** del corpus con el aprendizaje local de Word2Vec.

$$J = \sum_{i,j=1}^{V} f(X_{ij}) \left( w_i^T \tilde{w}_j + b_i + \tilde{b}_j - \log X_{ij} \right)^2$$

| | Word2Vec | GloVe |
|---|---|---|
| Tipo de aprendizaje | Predictivo (NN) | Basado en conteos |
| Información usada | Ventana local | Co-ocurrencias globales |
| Velocidad | Rápido | Más lento (construcción matriz) |
| Calidad analogías | Alta | Alta |

[paper GloVe](https://nlp.stanford.edu/pubs/glove.pdf)

### FastText

**FastText** extiende Word2Vec representando cada palabra como la **suma de sus n-gramas de caracteres**.

```
"where" con n=3:
n-gramas: ["<wh", "whe", "her", "ere", "re>", "<where>"]
embedding("where") = suma de embeddings de n-gramas
```

**Ventaja principal:** maneja palabras **fuera del vocabulario (OOV)**

```
Palabra OOV: "tokenizando"
FastText: usa "token", "okeni", "keniz", ... → puede estimar embedding
Word2Vec: ❌ sin embedding para palabras no vistas
```

In [ ]:
from gensim.models import FastText

# Entrenar FastText 
model_ft = FastText(
    sentences=sentences,
    vector_size=50,
    window=3,
    min_count=1,
    sg=1,
    min_n=2,      # mínimo tamaño de n-grama de caracteres
    max_n=4,      # máximo tamaño
    epochs=200,
    seed=42
)

print("FastText entrenado:")
print(f"  Vocabulario: {len(model_ft.wv)} palabras")

# Comparación con palabras OOV 
palabras_oov = ['tokenizando', 'preprocesador', 'clasificador', 'nlpista']

print("\nMANEJO DE PALABRAS OOV:")
print(f"  {'Palabra OOV':<20} {'Word2Vec':>12} {'FastText':>12}")
print("  " + "-"*46)
for p in palabras_oov:
    en_w2v = p in model_w2v.wv
    # FastText siempre puede generar un vector para palabras OOV
    try:
        vec_ft = model_ft.wv[p]
        ft_str = f"vec dim={len(vec_ft)}"
    except:
        ft_str = "No disponible"
    print(f"  {p:<20} {'Sí' if en_w2v else 'No (OOV)':>12} {ft_str:>12}")

In [ ]:
# Cargar embeddings GloVe pre-entrenados (simulado) 
# En producción se descarga desde: https://nlp.stanford.edu/projects/glove/
# Aquí simulamos un pequeño subset

glove_subset = {
    'king':   np.array([ 0.50,  0.69, -0.40,  0.34,  0.05]),
    'queen':  np.array([ 0.47,  0.64, -0.37,  0.32, -0.08]),
    'man':    np.array([ 0.30,  0.42, -0.20,  0.05,  0.60]),
    'woman':  np.array([ 0.28,  0.38, -0.18,  0.04, -0.55]),
    'prince': np.array([ 0.48,  0.65, -0.38,  0.30,  0.03]),
    'cat':    np.array([-0.30,  0.20,  0.50, -0.10,  0.10]),
    'dog':    np.array([-0.28,  0.22,  0.48, -0.12,  0.08]),
    'animal': np.array([-0.20,  0.25,  0.45, -0.08,  0.05]),
    'paris':  np.array([ 0.70,  0.20,  0.10,  0.60, -0.05]),
    'france': np.array([ 0.65,  0.15,  0.05,  0.62, -0.10]),
    'berlin': np.array([ 0.68,  0.18,  0.08,  0.58, -0.07]),
    'germany':np.array([ 0.63,  0.13,  0.03,  0.60, -0.12]),
}

def analogia_glove(a, b, c, embeddings, top_n=3):
    """Resuelve: a es a b como c es a ???"""
    # vec(b) - vec(a) + vec(c)
    target = embeddings[b] - embeddings[a] + embeddings[c]
    target = target / np.linalg.norm(target)
    excluir = {a, b, c}
    sims = []
    for w, v in embeddings.items():
        if w not in excluir:
            v_norm = v / np.linalg.norm(v)
            sims.append((w, np.dot(target, v_norm)))
    return sorted(sims, key=lambda x: -x[1])[:top_n]

print("ANALOGÍAS CON GloVe (simulado):")
print()
analogias = [
    ('man', 'king', 'woman'),      # man:king :: woman:???
    ('paris', 'france', 'berlin'), # paris:france :: berlin:???
    ('cat', 'animal', 'dog'),      # cat:animal :: dog:???
]
for a, b, c in analogias:
    resultado = analogia_glove(a, b, c, glove_subset)
    print(f"  '{a}' es a '{b}' como '{c}' es a → {[(w, round(s,3)) for w, s in resultado[:2]]}")

## 5. Sentence Embeddings 

### ¿Por qué necesitamos embeddings de oraciones?

Los embeddings de palabras (Word2Vec, GloVe) representan palabras individuales. Para comparar **documentos completos** necesitamos embeddings a nivel de oración.

### Enfoques

| Enfoque | Descripción | Calidad |
|---|---|---|
| Promedio de Word2Vec | Promedio de los vectores de palabras | Básica |
| TF-IDF weighted avg | Promedio ponderado por TF-IDF | Media |
| **Sentence-BERT** | Red siamesa sobre BERT | **Alta** |
| **Universal Sentence Encoder** | Transformer + DAN | Alta |
| **all-MiniLM** | MiniLM destilado | Alta + rápido |

📚 Documentación: https://www.sbert.net/

In [ ]:
import numpy as np

# Sentence embedding por promedio de Word2Vec 
def sentence_embedding_avg(oracion, modelo):
    """Promedia los embeddings de las palabras de la oración."""
    tokens = re.findall(r'[a-záéíóúüñ]+', oracion.lower())
    vecs = [modelo.wv[t] for t in tokens if t in modelo.wv]
    if not vecs:
        return np.zeros(modelo.vector_size)
    return np.mean(vecs, axis=0)

def similitud_coseno(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)

# Oraciones de prueba
oraciones = [
    "los modelos de lenguaje natural procesan texto",
    "el procesamiento de texto usa modelos de lenguaje",
    "python es un lenguaje de programación popular",
    "los algoritmos de machine learning aprenden de datos",
    "el aprendizaje profundo usa redes neuronales",
]

# Calcular embeddings
embs = [sentence_embedding_avg(s, model_w2v) for s in oraciones]

# Matriz de similitud
n = len(oraciones)
sim_mat = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        sim_mat[i, j] = similitud_coseno(embs[i], embs[j])

print("SIMILITUD ENTRE ORACIONES (promedio Word2Vec):")
print()
for i, s in enumerate(oraciones):
    print(f"  [{i+1}] {s}")
print()
print(f"  {'':5}" + ''.join(f'[{j+1}]   ' for j in range(n)))
for i in range(n):
    fila = f"  [{i+1}]  " + '  '.join(f'{sim_mat[i,j]:.3f}' for j in range(n))
    print(fila)

## 6. Ejercicio: Similitud Semántica 

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

#  Motor de búsqueda semántica con Word2Vec 
class BuscadorSemantico:
    """Búsqueda semántica usando embeddings promediados de Word2Vec."""

    def __init__(self, corpus, modelo):
    
    def buscar(self, query, top_n=3):
       
        return 

In [ ]:
# Corpus para búsqueda
corpus_busqueda = [
    "bert es un modelo de lenguaje pre-entrenado de google",
    "word2vec aprende embeddings de palabras con redes neuronales",
    "la tokenización divide el texto en unidades menores",
    "el análisis de sentimientos clasifica opiniones en positivas y negativas",
    "los transformers usan mecanismos de atención para procesar secuencias",
    "fasttext maneja palabras fuera del vocabulario con subpalabras",
    "spacy es una librería industrial para el procesamiento de lenguaje",
    "la similitud de coseno mide el ángulo entre vectores de palabras",
    "los modelos de deep learning aprenden características automáticamente",
    "gpt genera texto coherente usando predicción de siguiente token",
]


In [ ]:
#  Visualización comparativa de representaciones 
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# t-SNE de los embeddings del corpus
from sklearn.manifold import TSNE
from sklearn.feature_extraction.text import TfidfVectorizer

# Word2Vec embeddings
embs_corpus = np.array([sentence_embedding_avg(d, model_w2v) for d in corpus_busqueda])
if embs_corpus.shape[0] > 2:
    tsne = TSNE(n_components=2, random_state=42, perplexity=min(4, len(corpus_busqueda)-1))
    coords_w2v = tsne.fit_transform(embs_corpus)

    axes[0].scatter(coords_w2v[:, 0], coords_w2v[:, 1], c='steelblue', s=120)
    for i, doc in enumerate(corpus_busqueda):
        label = ' '.join(doc.split()[:3]) + '...'
        axes[0].annotate(label, coords_w2v[i], xytext=(3, 3),
                         textcoords='offset points', fontsize=7)
    axes[0].set_title('Sentence Embeddings — Word2Vec avg (t-SNE)', fontsize=10)
    axes[0].grid(True, alpha=0.3)

# TF-IDF embeddings
tv2 = TfidfVectorizer()
X_tfidf2 = tv2.fit_transform(corpus_busqueda).toarray()
tsne2 = TSNE(n_components=2, random_state=42, perplexity=min(4, len(corpus_busqueda)-1))
coords_tfidf = tsne2.fit_transform(X_tfidf2)

axes[1].scatter(coords_tfidf[:, 0], coords_tfidf[:, 1], c='tomato', s=120)
for i, doc in enumerate(corpus_busqueda):
    label = ' '.join(doc.split()[:3]) + '...'
    axes[1].annotate(label, coords_tfidf[i], xytext=(3, 3),
                     textcoords='offset points', fontsize=7)
axes[1].set_title('TF-IDF Vectors (t-SNE)', fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Comparativa de representaciones vectoriales (t-SNE)', fontsize=12)
plt.tight_layout()
plt.show()

## 7. Comparativa: BoW vs TF-IDF vs Embeddings 

| Característica | BoW | TF-IDF | Word2Vec | GloVe | FastText | SBERT |
|---|---|---|---|---|---|---|
| **Semántica** | ❌ | ❌ | ✅ | ✅ | ✅ | ✅✅ |
| **OOV** | ❌ | ❌ | ❌ | ❌ | ✅ | ✅ |
| **Contexto** | ❌ | ❌ | ❌ | ❌ | ❌ | ✅✅ |
| **Interpretable** | ✅✅ | ✅✅ | ❌ | ❌ | ❌ | ❌ |
| **Velocidad** | ✅✅ | ✅✅ | ✅ | ✅ | ✅ | ⚠️ |
| **Dimensiones** | |V| | |V| | 50-300 | 50-300 | 50-300 | 384-768 |
| **Multilingüe** | ❌ | ❌ | Depende | Depende | ✅ | ✅ |

### Regla de uso

```
Datos pequeños + necesito interpretabilidad  →  TF-IDF
Datos medianos + palabras raras              →  FastText
Datos grandes + analogías semánticas         →  Word2Vec / GloVe
Oraciones / párrafos enteros                 →  Sentence-BERT
Fine-tuning en tarea específica              →  BERT / RoBERTa
```